# SESION 2: Tarea de Spark · Caso de negocio con `samples.tpch` 

## 1. El dataset: `samples.tpch`

| Tabla | Qué es | Columnas que usaremos |
|---|---|---|
| `orders` | Un pedido por fila | `o_custkey`, `o_orderstatus`, `o_totalprice`, `o_orderdate` |
| `customer` | Un cliente por fila | `c_custkey`, `c_nationkey`, `c_mktsegment` |
| `nation` | Países (25) | `n_nationkey`, `n_name` |

Se conectan así: `orders.o_custkey` → `customer.c_custkey` y `customer.c_nationkey` → `nation.n_nationkey`.


In [0]:
from pyspark.sql import functions as F

orders   = spark.table("samples.tpch.orders")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")

## 3. Preguntas de negocio

### Pregunta 1 · Pedidos por estado 

¿Cuántos pedidos hay en cada estado (`o_orderstatus`) y cuál es el monto total de cada uno?

*Pista:* `groupBy` sobre `orders` con dos agregaciones: `count` y `sum` de `o_totalprice`.

In [0]:
resultado_1 = (orders
    .groupBy("o_orderstatus")
    .agg(F.count("*").alias("num_pedidos"),
         F.sum("o_totalprice").alias("monto_total"))
    .orderBy("o_orderstatus"))

display(resultado_1)

o_orderstatus,num_pedidos,monto_total
F,3654501,549181919365.27
O,3654310,548923692869.55
P,191189,35333603011.43


### Pregunta 2 · Evolución anual 
¿Cuántos pedidos y qué monto total hay por año (`o_orderdate`)? Ordena del año más antiguo al más reciente.

*Pista:* Crea la columna `año` con `F.year(...)` y agrupa por ella.

In [0]:
resultado_2 = (orders
    .withColumn("año", F.year("o_orderdate"))
    .groupBy("año")
    .agg(F.count("*").alias("num_pedidos"),
         F.sum("o_totalprice").alias("monto_total"))
    .orderBy("año"))

display(resultado_2)

año,num_pedidos,monto_total
1992,1139873,172177707475.98
1993,1138893,172229906369.71
1994,1137944,172079589599.59
1995,1136308,171632673720.20
1996,1141556,172605588830.66
1997,1137325,171861381581.98
1998,668101,100852367668.13


### Pregunta 3 · Pedidos grandes ya finalizados 

¿Cuántos pedidos finalizados (`o_orderstatus` = `'F'`) tienen un monto (`o_totalprice`) mayor a 100,000?

*Pista:* Usa `filter` con las dos condiciones unidas con `&` (cada una entre paréntesis) y luego `agg(F.count("*"))`.

In [0]:
resultado_3 = (orders
    .filter((F.col("o_orderstatus") == "F") & (F.col("o_totalprice") > 100000))
    .agg(F.count("*").alias("pedidos_grandes_finalizados")))

display(resultado_3)

pedidos_grandes_finalizados
2418589


### Pregunta 4 · Clientes por segmento

¿Cuántos clientes hay en cada segmento de mercado (`c_mktsegment`)? Ordénalos de mayor a menor.

*Pista:* `groupBy` sobre `customer` y `count`.

In [0]:
resultado_4 = (customer
    .groupBy("c_mktsegment")
    .agg(F.count("*").alias("num_clientes"))
    .orderBy(F.desc("num_clientes")))

display(resultado_4)

c_mktsegment,num_clientes
BUILDING,150297
FURNITURE,150190
MACHINERY,150165
HOUSEHOLD,149687
AUTOMOBILE,149661


### Pregunta 5 · Clientes por país

¿En qué 5 países (`n_name`) hay más clientes? Muestra el país y el número de clientes, de mayor a menor.

*Pista:* Une `customer` con `nation` (`c_nationkey` = `n_nationkey`), agrupa por `n_name` y cuenta.

In [0]:
resultado_5 = (customer
    .join(nation, customer.c_nationkey == nation.n_nationkey)
    .groupBy("n_name")
    .agg(F.count("*").alias("num_clientes"))
    .orderBy(F.desc("num_clientes"))
    .limit(5))

display(resultado_5)

n_name,num_clientes
INDONESIA,30401
IRAN,30257
INDIA,30234
IRAQ,30232
ETHIOPIA,30201


### Pregunta 6 · Ticket promedio por segmento

¿Cuál es el ticket promedio (monto promedio por pedido, `o_totalprice`) de cada segmento de mercado? Ordénalo de mayor a menor.

*Pista:* Une `orders` con `customer` (`o_custkey` = `c_custkey`), agrupa por `c_mktsegment` y usa `F.avg`.

In [0]:
resultado_6 = (orders
    .join(customer, orders.o_custkey == customer.c_custkey)
    .groupBy("c_mktsegment")
    .agg(F.round(F.avg("o_totalprice"), 2).alias("ticket_promedio"))
    .orderBy(F.desc("ticket_promedio")))

display(resultado_6)

c_mktsegment,ticket_promedio
FURNITURE,151162.89
BUILDING,151145.72
MACHINERY,151127.45
HOUSEHOLD,151112.32
AUTOMOBILE,151077.38
